# ComplaintSense: Consumer Complaint Classification — Starter

Welcome! In this competition, you will teach a machine-learning model to read a customer complaint and route it to one of ten support categories.

### What you will learn

- How to inspect labelled text data.
- How to create a local validation set.
- How TF-IDF converts text into numeric features.
- How to measure multiclass performance with Balanced F1 (macro F1).
- How to fine-tune a pretrained DistilBERT model attached through Kaggle Models.
- How to create, validate, and submit a Kaggle competition file.

### Two modelling paths

1. **Beginner baseline:** character TF-IDF + logistic regression. This is fast and works on CPU.
2. **Pretrained transformer:** DistilBERT fine-tuning. This is more advanced and should use a GPU.

> **Submission rule:** You must use a Kaggle Notebook. Run it from top to bottom, choose **Save Version → Save & Run All**, and submit `/kaggle/working/submission.csv` from the saved version's **Output** panel.

## Understand the files and target

The competition provides two CSV files:

- `train_complaints.csv` contains `ComplaintId`, complaint `text`, the correct `Category`, and a training-only `FamilyId`.
- `test_complaints.csv` contains `ComplaintId` and `text`, but hides `Category`. Your model must predict it.

`ComplaintId` is only an identifier; do not use it as a language feature. `text` is the model input. `Category` is the answer (target label). `FamilyId` marks complaints derived from the same semantic scenario. It is used only to prevent leakage during local validation; it is not a prediction feature and does not appear in the test file.

The leaderboard uses **Balanced F1**, configured as macro F1. Kaggle calculates F1 separately for every category and then averages the ten scores. Each category therefore contributes equally, even if some categories have fewer examples.

In [ ]:
# pathlib makes file paths easier to work with.
from pathlib import Path
import pandas as pd

# Kaggle mounts competition files as read-only inputs in this directory.
base = "/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/"
train = pd.read_csv(base + 'train_complaints.csv')
test = pd.read_csv(base + 'test_complaints.csv')

# Fail early with a helpful message if the wrong files were attached.
assert list(train.columns) == ['ComplaintId', 'text', 'Category', 'FamilyId']
assert list(test.columns) == ['ComplaintId', 'text']
assert train['text'].notna().all() and test['text'].notna().all()

print(len(train), 'train ·', len(test), 'test')
print('Categories:', train['Category'].nunique())
display(train.head())
display(train['Category'].value_counts().rename('training examples'))

## 1. Build a TF-IDF baseline

Machine-learning models cannot read raw sentences directly. A **TF-IDF vectorizer** turns pieces of text into numeric features. This baseline uses character groups of length 3–5, called character n-grams. They are useful for short complaints because they can recognise word fragments, spelling variations, and related forms such as `charge`, `charged`, and `charging`.

**Logistic regression** learns which TF-IDF patterns are associated with each category. Despite its name, it is a classification algorithm. `class_weight='balanced'` gives more training importance to categories with fewer rows.

Before using all training data, we hold out entire semantic families for **local validation**. A normal random row split could place closely related variations of the same scenario on both sides and produce an unrealistically high score. `StratifiedGroupKFold` keeps every `FamilyId` entirely on one side while preserving all ten categories in the selected fold.

A local score and the leaderboard score can still differ because they contain different held-out families. Use validation to compare ideas—not as a guaranteed leaderboard result.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

# Reserve one of four folds (about 25%) for validation.
# y preserves class balance; groups prevents a FamilyId crossing the boundary.
group_splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
fit_index, valid_index = next(group_splitter.split(
    train,
    y=train['Category'],
    groups=train['FamilyId'],
))
baseline_train = train.iloc[fit_index].copy()
baseline_valid = train.iloc[valid_index].copy()

assert set(baseline_train['FamilyId']).isdisjoint(set(baseline_valid['FamilyId']))
assert baseline_train['Category'].nunique() == baseline_valid['Category'].nunique() == 10
print('Fit rows and families:', len(baseline_train), baseline_train['FamilyId'].nunique())
print('Validation rows and families:', len(baseline_valid), baseline_valid['FamilyId'].nunique())

# A Pipeline guarantees that validation/test text receives the same transformation.
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])

# Fit only on the baseline training portion, then evaluate unseen validation rows.
pipe.fit(baseline_train['text'], baseline_train['Category'])
valid_pred = pipe.predict(baseline_valid['text'])
valid_f1 = f1_score(
    baseline_valid['Category'],
    valid_pred,
    average='macro',
    labels=sorted(train['Category'].unique()),
    zero_division=0,
)
print(f'Local validation Balanced F1: {valid_f1:.4f}')

# Once the approach is chosen, refit on every labelled row before test prediction.
pipe.fit(train['text'], train['Category'])
pred = pipe.predict(test['text'])

## 2. Build and validate the Kaggle submission

Kaggle requires one prediction per test row, in the original test order, with exactly the columns `ComplaintId` and `Category`.

The assertions below are safety checks. If one fails, read its condition before submitting. They prevent common mistakes such as an extra index column, missing rows, duplicated IDs, shuffled IDs, blank predictions, or misspelled category names.

`/kaggle/input` is read-only. Generated files must be written to `/kaggle/working`.

In [ ]:
# Only these labels are accepted by the competition scorer.
valid_categories = set(train['Category'].astype(str).unique())

# Preserve ComplaintId directly from test so predictions stay aligned with test rows.
submission = pd.DataFrame({
    'ComplaintId': test['ComplaintId'].values,
    'Category': pred,
})

# Validate the complete submission contract before writing the file.
assert list(submission.columns) == ['ComplaintId', 'Category'], 'Wrong columns or order.'
assert len(submission) == len(test)
assert submission['ComplaintId'].tolist() == test['ComplaintId'].tolist()
assert submission['ComplaintId'].is_unique
assert submission['Category'].notna().all()
assert set(submission['Category']).issubset(valid_categories)

# index=False prevents pandas from adding an unwanted third column.
output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
print(f'Wrote {output_path} with {len(submission)} predictions')
display(submission.head())

## Submit the baseline, or continue to fine-tuning

The cell above creates a valid baseline submission. You may submit it now, or continue to the pretrained-model section below and let the final transformer cell replace `submission.csv`.

When your chosen approach has finished:

1. Confirm the latest submission cell reports **160 predictions**.
2. Click **Save Version** in the upper-right corner.
3. Select **Save & Run All** so Kaggle executes a clean committed copy from top to bottom.
4. Wait until the committed version finishes without errors.
5. Open that version's **Output** panel and confirm `submission.csv` is listed.
6. Click **Submit to Competition**, select `submission.csv` if prompted, and add a short description of the model.
7. Keep the committed notebook version and share its link with the organizers if requested.

Do not submit an interactive draft that has not completed **Save & Run All**. Do not add the private solution file or hard-code hidden labels.

## 3. Fine-tune a pretrained model from Kaggle Models

Kaggle can attach pretrained weights directly to a notebook, so no internet download is needed during the committed run:

1. In the notebook editor, open the right-hand **Input** panel.
2. Click **Add Input**, choose **Models**, and search for **DistilBERT base uncased**.
3. Add a Hugging Face Transformers/PyTorch variation containing `config.json`, tokenizer files, and either `model.safetensors` or `pytorch_model.bin`.
4. For faster fine-tuning, open **Settings → Accelerator** and select a GPU.

The next cell finds the attached model under `/kaggle/input`. Attach only one compatible Hugging Face base model, or set `MODEL_PATH` manually if you attach several.

### What fine-tuning means

DistilBERT was pretrained on a large collection of general English text. It already understands useful language patterns, but it does not know the ten ComplaintSense categories. Fine-tuning adds a ten-class prediction layer and adjusts the pretrained weights using our labelled complaints. We use a small learning rate so the model learns the new task without rapidly forgetting its pretrained language knowledge.

In [ ]:
import os
from pathlib import Path

# Avoid harmless tokenizer worker warnings in the notebook output.
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Kaggle may generate the attached model's directory name. Find the model by
# looking for the configuration, tokenizer, and weight files it must contain.
model_candidates = []
for config_file in Path('/kaggle/input').rglob('config.json'):
    folder = config_file.parent
    has_tokenizer = (folder / 'vocab.txt').exists() or (folder / 'tokenizer.json').exists()
    has_weights = (folder / 'model.safetensors').exists() or (folder / 'pytorch_model.bin').exists()
    if has_tokenizer and has_weights:
        model_candidates.append(folder)

if not model_candidates:
    raise FileNotFoundError(
        'No attached Hugging Face model found. Use Add Input → Models and attach '
        'DistilBERT base uncased before running this section.'
    )

MODEL_PATH = str(sorted(model_candidates, key=lambda p: len(str(p)))[0])
print('Using pretrained model:', MODEL_PATH)

### Prepare and train DistilBERT

The next cell has seven main jobs:

1. **Encode labels:** map category strings to numeric IDs 0–9 and back again.
2. **Split data:** reuse the baseline's group-aware split, which reserves about 25% of semantic families for validation.
3. **Tokenize:** convert complaint text into token IDs DistilBERT understands. Complaints longer than 128 tokens are truncated.
4. **Wrap datasets:** provide tokenized rows to PyTorch one at a time.
5. **Load the model:** reuse pretrained language weights and create a new ten-class output layer.
6. **Measure Balanced F1:** convert the model's ten output scores to class predictions and calculate macro F1.
7. **Fine-tune:** train for four epochs with a small `2e-5` learning rate.

These settings are sensible starting points, not guaranteed best values. Trying justified alternatives is part of the competition. GPU training is recommended.

In [ ]:
import inspect
import numpy as np
import torch
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

# Neural networks use numeric labels; retain both conversion directions.
labels = sorted(train['Category'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
# Reuse the same leakage-resistant family split so models are compared fairly.
train_part = baseline_train.copy()
valid_part = baseline_valid.copy()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

# Tokenize an example only when the Trainer requests it.
class ComplaintDataset(Dataset):
    def __init__(self, frame, include_labels=True):
        self.frame = frame.reset_index(drop=True)
        self.include_labels = include_labels

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        item = tokenizer(
            str(row['text']),
            truncation=True,
            max_length=128,
        )
        if self.include_labels:
            item['labels'] = label2id[row['Category']]
        return item

train_dataset = ComplaintDataset(train_part)
valid_dataset = ComplaintDataset(valid_part)
test_dataset = ComplaintDataset(test, include_labels=False)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

def compute_metrics(evaluation):
    # Each row has ten logits. argmax selects the largest as the predicted ID.
    predicted_ids = np.argmax(evaluation.predictions, axis=-1)
    return {
        'balanced_f1': f1_score(
            evaluation.label_ids,
            predicted_ids,
            average='macro',
            labels=list(range(len(labels))),
            zero_division=0,
        )
    }

# Transformers renamed the evaluation-strategy argument; support both versions.
argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
strategy_name = 'eval_strategy' if 'eval_strategy' in argument_parameters else 'evaluation_strategy'
training_kwargs = {
    'output_dir': '/kaggle/working/distilbert-complaints',
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'num_train_epochs': 15,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'balanced_f1',
    'greater_is_better': True,
    'logging_steps': 10,
    'warmup_ratio': 0.1,
    'report_to': 'none',
    'seed': 42,
    'fp16': False,
}
training_kwargs[strategy_name] = 'epoch'
training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
    'eval_dataset': valid_dataset,
    'compute_metrics': compute_metrics,
}
if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
# Fine-tune, then report validation loss and Balanced F1.
trainer.train()
results = trainer.evaluate()
results

### Create the transformer submission

`trainer.predict` produces ten scores for every hidden-label test complaint. We select the largest score, convert the numeric class ID back to its category name, and run the same submission checks used by the baseline.

This cell overwrites the earlier baseline file. After it runs, `/kaggle/working/submission.csv` contains the DistilBERT predictions.

In [ ]:
# Generate one set of ten class scores for every test complaint.
transformer_output = trainer.predict(test_dataset)
transformer_ids = np.argmax(transformer_output.predictions, axis=-1)
transformer_labels = [id2label[int(index)] for index in transformer_ids]

submission = pd.DataFrame({
    'ComplaintId': test['ComplaintId'].values,
    'Category': transformer_labels,
})

assert list(submission.columns) == ['ComplaintId', 'Category']
assert len(submission) == len(test) == 160
assert submission['ComplaintId'].tolist() == test['ComplaintId'].tolist()
assert submission['ComplaintId'].is_unique
assert submission['Category'].notna().all()
assert set(submission['Category']).issubset(set(labels))

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Transformer submission written to /kaggle/working/submission.csv')
display(submission.head())

In [ ]:
# Get predictions on the validation set
validation_output = trainer.predict(valid_dataset)
validation_pred_ids = np.argmax(validation_output.predictions, axis=-1)
validation_pred_labels = [id2label[int(index)] for index in validation_pred_ids]
validation_true_labels = valid_dataset.frame['Category'].tolist()

# Build a DataFrame from the validation dataset
validation_results = pd.DataFrame({
    "true_label": validation_true_labels,
    "predicted_label": validation_pred_labels,
})

# Find incorrect predictions
errors = validation_results[
    validation_results["true_label"] != validation_results["predicted_label"]
].copy()

print(f"Validation examples: {len(validation_results)}")
print(f"Errors: {len(errors)}")

display(errors.head(20))

In [ ]:
error_by_category = (
    errors["true_label"]
    .value_counts()
    .rename("errors")
)

display(error_by_category)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
cm = confusion_matrix(
    validation_true_labels,
    validation_pred_labels,
    labels=list(labels)
)

fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(labels)
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format='d'
)

plt.title("Validation Set Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
error_by_category = (
    validation_results[
        validation_results["true_label"] != validation_results["predicted_label"]
    ]["true_label"]
    .value_counts()
    .reindex(labels, fill_value=0)
)

plt.figure(figsize=(12, 6))
error_by_category.plot(kind="bar")

plt.title("Validation Errors by Actual Category")
plt.xlabel("Actual Category")
plt.ylabel("Number of Errors")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Show the most common misclassification pairs
misclassification_pairs = (
    errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(misclassification_pairs)

In [ ]:
category_total = validation_results["true_label"].value_counts()

category_errors = (
    errors["true_label"]
    .value_counts()
)

error_analysis = pd.DataFrame({
    "total_examples": category_total,
    "errors": category_errors
}).fillna(0)

error_analysis["correct"] = (
    error_analysis["total_examples"] - error_analysis["errors"]
)

error_analysis["error_rate_%"] = (
    error_analysis["errors"] /
    error_analysis["total_examples"] * 100
).round(1)

error_analysis = error_analysis.reindex(labels)

display(error_analysis)

In [ ]:
# Inspect the actual complaints the model got wrong
worst_categories = [
    "customer_service",
    "delivery_shipping",
    "subscription_cancel"
]

# Get the validation predictions
validation_output = trainer.predict(valid_dataset)

validation_pred_ids = np.argmax(
    validation_output.predictions,
    axis=-1
)

validation_pred_labels = [
    id2label[int(index)]
    for index in validation_pred_ids
]

# Build error analysis table with the original complaint text
validation_results = valid_dataset.frame.copy()

validation_results["true_label"] = validation_results["Category"]
validation_results["predicted_label"] = validation_pred_labels

errors = validation_results[
    validation_results["true_label"] != validation_results["predicted_label"]
].copy()

# Focus on the three categories causing most errors
worst_errors = errors[
    errors["true_label"].isin(worst_categories)
][["text", "true_label", "predicted_label"]]

print(f"Total validation errors: {len(errors)}")
print(f"Errors in the three worst categories: {len(worst_errors)}")

display(worst_errors)

## Beginner troubleshooting

| Problem | What to check |
|---|---|
| Competition CSV `FileNotFoundError` | Join the competition and confirm its data appears in the notebook **Input** panel. Keep the supplied `base` path unchanged. |
| No attached Hugging Face model found | Use **Add Input → Models** and attach a Transformers/PyTorch DistilBERT model with configuration, tokenizer, and weight files. |
| CUDA out-of-memory | Reduce `per_device_train_batch_size` from 16 to 8 or 4, then restart the session. |
| Training is very slow | Enable a GPU under **Settings → Accelerator**. |
| Low local F1 | Inspect errors by category and carefully test learning rate, epochs, or model choice using validation data only. |
| Missing `submission.csv` | Run the latest prediction cell and confirm it writes to `/kaggle/working/submission.csv`. |
| Kaggle rejects the CSV | Check for 160 rows, exact columns `ComplaintId,Category`, unique IDs, valid labels, and `index=False`. |

### Responsible improvement ideas

- Plot a confusion matrix to discover which complaint categories are confused.
- Compare word TF-IDF, character TF-IDF, and transformer errors.
- Tune hyperparameters systematically and record each experiment.
- Try ensembling only after understanding the strengths of individual models.
- Keep random seeds and explanations so another student can reproduce your work.

Never use private solution files, manually label hidden test rows, or hard-code answers. The objective is to build a model that generalises.